# 05 – Kostenanalyse: Was die Unzuverlässigkeit kostet und was sie zu beheben kostet

Dieses Notebook bewertet die in den Notebooks 01 bis 04 gemessenen Befunde
wirtschaftlich und vergleicht Handlungsoptionen.

| Abschnitt | Inhalt |
|-----------|--------|
| 1 | Was die gemessene Verspätung kostet — in Fahrzeugstunden und Euro |
| 2 | Die drei Maßnahmen im Vergleich: Abfahrtsdisziplin, Signaltechnik, Gleisbau |
| 3 | U-Bahn gegen Tram: Baukosten und Kapazität je Kilometer |
| 4 | Historischer Kontext: warum früher so viel U-Bahn gebaut wurde |
| 5 | Resilienz: was die Wetterabhängigkeit der Tram bedeutet |

---

### Vorbemerkung zur Belastbarkeit der Zahlen

Dieses Notebook verlässt an mehreren Stellen den durch die eigenen Messungen gedeckten
Bereich. Damit erkennbar bleibt, worauf eine Zahl beruht, ist jede Angabe einer von drei
Stufen zugeordnet:

| Stufe | Bedeutung |
|-------|-----------|
| **gemessen** | aus den erhobenen Daten berechnet (Notebooks 01–04) |
| **Kostensatz** | externer Standardwert aus Literatur oder Tarifwerk |
| **Annahme** | gesetzte Größe, die die Analyse nicht belegen kann |

> **Trennung der Kostenarten.** Direkte Budgetkosten (Fahrpersonal der BVG) und
> volkswirtschaftliche Kosten (Zeitverlust der Fahrgäste) werden durchgängig **getrennt
> ausgewiesen**. Volkswirtschaftliche Beträge sind kein Haushaltsposten der BVG und
> dürfen nicht mit ihnen verrechnet werden.

> **Korrektur gegenüber einer früheren Fassung.** Dieses Notebook rechnete zuvor mit der
> Median-Differenz aus Hypothese H6b (Notebook 03) und bezeichnete sie als
> „Mehrverspätung pro Fahrgastfahrt". Beides war unzutreffend: Der Test in H6b ist
> zirkulär konstruiert (Notebook 03, Abschnitt 4c), und die Größe beschreibt eine
> Differenz zwischen Haltestellenmittelwerten, nicht den Zeitverlust einer einzelnen
> Fahrgastfahrt. Die Rechnung stützt sich jetzt auf die **gemessene erzeugte Verspätung**
> aus Notebook 04.

## 0 – Setup

In [1]:
import warnings, sys, pathlib
warnings.filterwarnings("ignore")

try:
    _cwd = pathlib.Path.cwd()
except (FileNotFoundError, OSError):
    _cwd = pathlib.Path(globals().get("__vsc_ipynb_file__", "~")).expanduser().resolve().parent
_root = _cwd
while not (_root / "config").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from elasticsearch import Elasticsearch

from config.settings import ES_HOST, ES_USER, ES_PASSWORD
from src.analysis.segmente import (
    segmente_gesamtzeitraum, MIN_BEOBACHTUNGEN_JE_SEGMENT,
)
from src.analysis.quality import ANALYSE_START, ANALYSE_ENDE

es = Elasticsearch(ES_HOST, basic_auth=(ES_USER, ES_PASSWORD), request_timeout=900)
INDEX = "tram-departures-v2"
assert es.ping(), "Elasticsearch nicht erreichbar"
print("Setup OK")

Setup OK


---
## 1 – Was die gemessene Verspätung kostet

Grundlage ist die in Notebook 04 berechnete **erzeugte Verspätung**: die Summe aller
Verspätungszuwächse zwischen zwei Haltestellen, über den **gesamten Erhebungszeitraum**
(60 Werktage, rund 7,1 Mio. Segmentbeobachtungen).

Diese Größe hat gegenüber der früher verwendeten H6b-Differenz drei Vorteile:

- Sie ist **direkt gemessen**, nicht aus einem Hypothesentest abgeleitet.
- Sie hat eine **eindeutige Einheit**: Fahrzeugsekunden, die im Betrieb zusätzlich
  anfallen.
- Sie ist **additiv** — Abschnitte lassen sich zu Linien und zum Netz aufsummieren.

Ausgewiesen werden zwei Varianten:

| Größe | Bedeutung |
|-------|-----------|
| **brutto** | Summe aller Verspätungszuwächse — der Aufwand, der im Betrieb entsteht |
| **netto** | brutto abzüglich der auf Recovery-Abschnitten wieder abgebauten Verspätung |

Für die Bewertung von Maßnahmen ist die **Bruttogröße** maßgeblich: Eine Verspätung, die
später wieder aufgeholt wird, hat trotzdem stattgefunden und kostet die Fahrgäste
dazwischen Zeit. Die Nettogröße beschreibt, was am Linienende übrig bleibt.

> Die Hochrechnung auf ein Jahr erfolgt über Werktage. **Wochenenden sind nicht
> enthalten** und werden nicht hochgerechnet; die Jahreszahl ist damit eine Untergrenze.

In [2]:
# ── Gemessene Grundlage: Segmentaggregate über den Gesamtzeitraum ────────────
CACHE = str(_root / "data" / "processed" / "segmente_tram_gesamt.parquet")
segment_gesamt = segmente_gesamtzeitraum(
    es, INDEX, von=ANALYSE_START, bis=ANALYSE_ENDE, cache_pfad=CACHE,
)

# Nur belastbare Abschnitte (Begründung siehe NB 04, Abschnitt 4)
belastbar = segment_gesamt[segment_gesamt["n"] >= MIN_BEOBACHTUNGEN_JE_SEGMENT]

brutto_s = belastbar["summe_positiv"].sum()
netto_s  = belastbar.loc[belastbar["summe_delta"] > 0, "summe_delta"].sum()
n_tage   = segment_gesamt.attrs.get("n_tage", 60)

WERKTAGE_JE_JAHR = 250   # [Annahme] Werktage abzüglich Feiertage

brutto_h_je_werktag = brutto_s / 3600 / n_tage
brutto_h_je_jahr    = brutto_h_je_werktag * WERKTAGE_JE_JAHR
netto_h_je_jahr     = netto_s / 3600 / n_tage * WERKTAGE_JE_JAHR

print(f"Auswertungsgrundlage: {n_tage} Werktage, "
      f"{segment_gesamt['n'].sum():,} Segmentbeobachtungen")
print(f"Belastbare Abschnitte (n ≥ {MIN_BEOBACHTUNGEN_JE_SEGMENT:,}): {len(belastbar):,}\n")
print(f"Erzeugte Verspätung brutto: {brutto_s/3600:9,.0f} Fahrzeugstunden   [gemessen]")
print(f"                     netto: {netto_s/3600:9,.0f} Fahrzeugstunden   [gemessen]")
print()
print(f"Brutto je Werktag:          {brutto_h_je_werktag:9,.0f} Fahrzeugstunden")
print(f"Hochrechnung Jahr (brutto): {brutto_h_je_jahr:9,.0f} Fahrzeugstunden   "
      f"[{WERKTAGE_JE_JAHR} Werktage, ohne Wochenenden]")

Lade zwischengespeichertes Ergebnis: /Users/valeriamuggironi/Documents/Master/Semester 2/NoSQL/Semesterprojekt/Berlin-Tram-UBahn/data/processed/segmente_tram_gesamt.parquet
Auswertungsgrundlage: 60 Werktage, 7,145,773 Segmentbeobachtungen
Belastbare Abschnitte (n ≥ 2,000): 763

Erzeugte Verspätung brutto:    14,176 Fahrzeugstunden   [gemessen]
                     netto:     6,879 Fahrzeugstunden   [gemessen]

Brutto je Werktag:                236 Fahrzeugstunden
Hochrechnung Jahr (brutto):    59,068 Fahrzeugstunden   [250 Werktage, ohne Wochenenden]


In [3]:
# ── Kostensätze ──────────────────────────────────────────────────────────────
# Fahrpersonal: direkter Budgetposten der BVG
FAHRER_EUR_H_MIN = 25.00     # [Kostensatz] TVöD VKA EG 6 Stufe 3 (2024), Vollkosten
FAHRER_EUR_H_MAX = 30.00     # [Kostensatz] TVöD VKA EG 7 Stufe 4 (2024), Vollkosten
FAHRER_EUR_H_MID = (FAHRER_EUR_H_MIN + FAHRER_EUR_H_MAX) / 2

# Fahrgastzeit: volkswirtschaftlicher Wohlfahrtsverlust, KEIN BVG-Budgetposten
ZEITWERT_EUR_H_MIN = 12.00   # [Kostensatz] BMVI, Bewertungsverfahren BVWP 2030, ÖPNV
ZEITWERT_EUR_H_MAX = 15.00   # [Kostensatz] dto., Obergrenze
ZEITWERT_EUR_H_MID = (ZEITWERT_EUR_H_MIN + ZEITWERT_EUR_H_MAX) / 2

# Besetzungsgrad: wie viele Fahrgäste sind im Mittel an Bord?
# Diese Größe ist NICHT gemessen — die Daten enthalten keine Fahrgastzahlen.
# Sie wird deshalb als Spanne geführt und die Ergebnisse als Sensitivität gezeigt.
BESETZUNG_MIN = 30           # [Annahme] Schwachlast
BESETZUNG_MID = 60           # [Annahme] Tagesmittel über alle Linien
BESETZUNG_MAX = 100          # [Annahme] Hauptverkehrszeit

kosten_fahrer = pd.DataFrame([
    {"Ansatz": f"{s:,.2f} €/h", "Kostenart": "Budget (direkt)",
     "Jahreskosten €": brutto_h_je_jahr * s}
    for s in (FAHRER_EUR_H_MIN, FAHRER_EUR_H_MID, FAHRER_EUR_H_MAX)
])

kosten_fahrgast = pd.DataFrame([
    {"Besetzung": b, "Zeitwert €/h": z,
     "Personenstunden/Jahr": brutto_h_je_jahr * b,
     "Jahreskosten €": brutto_h_je_jahr * b * z}
    for b in (BESETZUNG_MIN, BESETZUNG_MID, BESETZUNG_MAX)
    for z in (ZEITWERT_EUR_H_MIN, ZEITWERT_EUR_H_MID, ZEITWERT_EUR_H_MAX)
])

print("A) Direkte Budgetkosten — Fahrpersonalzeit\n")
print(kosten_fahrer.to_string(index=False,
      formatters={"Jahreskosten €": lambda v: f"{v:,.0f}"}))

print("\n\nB) Volkswirtschaftlicher Zeitverlust der Fahrgäste")
print("   ACHTUNG: kein BVG-Budgetposten. Besetzungsgrad ist eine Annahme.\n")
print(kosten_fahrgast.to_string(index=False,
      formatters={"Personenstunden/Jahr": lambda v: f"{v:,.0f}",
                  "Jahreskosten €": lambda v: f"{v:,.0f}"}))

BUDGET_JAHR = brutto_h_je_jahr * FAHRER_EUR_H_MID
VWL_JAHR    = brutto_h_je_jahr * BESETZUNG_MID * ZEITWERT_EUR_H_MID
print(f"\n\nMittlere Ansätze:")
print(f"  Budgetkosten (Fahrpersonal):        {BUDGET_JAHR:12,.0f} € / Jahr")
print(f"  Volkswirtschaftlich (Fahrgastzeit): {VWL_JAHR:12,.0f} € / Jahr")
print(f"  Verhältnis:                         {VWL_JAHR/BUDGET_JAHR:.0f} : 1")

A) Direkte Budgetkosten — Fahrpersonalzeit

   Ansatz       Kostenart Jahreskosten €
25.00 €/h Budget (direkt)      1,476,688
27.50 €/h Budget (direkt)      1,624,356
30.00 €/h Budget (direkt)      1,772,025


B) Volkswirtschaftlicher Zeitverlust der Fahrgäste
   ACHTUNG: kein BVG-Budgetposten. Besetzungsgrad ist eine Annahme.

 Besetzung  Zeitwert €/h Personenstunden/Jahr Jahreskosten €
        30          12.0            1,772,025     21,264,300
        30          13.5            1,772,025     23,922,338
        30          15.0            1,772,025     26,580,375
        60          12.0            3,544,050     42,528,600
        60          13.5            3,544,050     47,844,675
        60          15.0            3,544,050     53,160,750
       100          12.0            5,906,750     70,881,000
       100          13.5            5,906,750     79,741,125
       100          15.0            5,906,750     88,601,250


Mittlere Ansätze:
  Budgetkosten (Fahrpersonal):          

In [4]:
# Sensitivität: wie stark hängt das Ergebnis an der Besetzungsannahme?
gitter = pd.DataFrame(
    [[brutto_h_je_jahr * b * z / 1e6 for z in (ZEITWERT_EUR_H_MIN, ZEITWERT_EUR_H_MID,
                                              ZEITWERT_EUR_H_MAX)]
     for b in (BESETZUNG_MIN, BESETZUNG_MID, BESETZUNG_MAX)],
    index=[f"{b} Fahrgäste" for b in (BESETZUNG_MIN, BESETZUNG_MID, BESETZUNG_MAX)],
    columns=[f"{z:.0f} €/h" for z in (ZEITWERT_EUR_H_MIN, ZEITWERT_EUR_H_MID,
                                      ZEITWERT_EUR_H_MAX)],
)

fig = px.imshow(
    gitter, text_auto=".1f", color_continuous_scale="OrRd", aspect="auto",
    labels={"x": "Zeitwert je Personenstunde", "y": "Angenommene Besetzung",
            "color": "Mio. €/Jahr"},
    title="Volkswirtschaftlicher Zeitverlust je Jahr (Mio. €) — Sensitivität der Annahmen",
)
fig.update_layout(height=380)
fig.show()

print("Die Spanne von Faktor", f"{gitter.values.max()/gitter.values.min():.1f}",
      "zwischen bester und schlechtester Annahme zeigt:")
print("Diese Zahl ist eine Größenordnung, kein Betrag.")

Die Spanne von Faktor 4.2 zwischen bester und schlechtester Annahme zeigt:
Diese Zahl ist eine Größenordnung, kein Betrag.


### Interpretation

**Die direkte Budgetwirkung ist klein, die volkswirtschaftliche groß.** Rund 1,6 Mio. €
zusätzliche Personalkosten stehen etwa 48 Mio. € verlorener Fahrgastzeit gegenüber — ein
Verhältnis von rund **1 : 29**. Das ist kein Rechenfehler, sondern die Folge davon, dass
in einem Fahrzeug ein Fahrer und mehrere Dutzend Fahrgäste sitzen.

Daraus folgt das **Kernproblem öffentlicher Investitionsrechnung**: Eine Maßnahme, die
sich aus Sicht des Verkehrsbetriebs kaum rechnet, kann gesamtwirtschaftlich hoch
rentabel sein. Wer nur den Haushalt der BVG betrachtet, unterschätzt den Nutzen von
Pünktlichkeitsmaßnahmen um etwa den Faktor dreißig.

**Was diese Zahlen nicht sind.** Der volkswirtschaftliche Betrag hängt vollständig an
einer Größe, die diese Arbeit nicht gemessen hat — dem Besetzungsgrad. Die
Sensitivitätsdarstellung zeigt eine Spanne von Faktor 4,2 zwischen den plausiblen
Randannahmen (21 bis 89 Mio. €). Belastbar sind deshalb die **Größenordnung und das
Verhältnis** der beiden Kostenarten, nicht der absolute Betrag.

**Drei Gründe, warum auch das noch eine Untergrenze ist:**

- Nur **Werktage** sind erfasst; Wochenenden werden nicht hochgerechnet.
- Nur **Verspätungen** sind enthalten. Die in Notebook 01 gemessenen **Verfrühungen**
  — 19 % aller Abfahrten — fehlen, obwohl sie Fahrgäste im Zweifel den vollen Takt kosten.
- Nur Abschnitte mit mindestens 2.000 Beobachtungen zählen (98,8 % aller Beobachtungen,
  aber nicht alle).

### Prüfung der Besetzungsannahme: spielt die Tageszeit eine Rolle?

Die Rechnung oben setzt eine **über alle Stunden konstante Besetzung** an. Das ist
angreifbar, denn nachts fahren erheblich weniger Menschen mit als in der Hauptverkehrszeit.

Ob das die Gesamtsumme verzerrt, hängt davon ab, **wann die Verspätung entsteht**:

- Entsteht sie überwiegend nachts, überschätzt die flache Annahme die Personenstunden.
- Entsteht sie überwiegend in der HVZ, unterschätzt sie sie.

Aus Notebook 04 (H4) ist bereits bekannt, dass die Nachtstunden im Mittel **genauso
verspätet** sind wie die Hauptverkehrszeit — die Vermutung liegt also nahe, dass ein
relevanter Teil der Verspätung anfällt, wenn kaum jemand mitfährt. Das lässt sich
direkt messen.

In [5]:
# Verspätungsentstehung je Tagesstunde (Stichprobe aus fünf Werktagen)
from src.analysis.segmente import lade_fahrten

STICHPROBENTAGE = [("2026-05-05", "2026-05-06"), ("2026-05-12", "2026-05-13"),
                   ("2026-05-19", "2026-05-20"), ("2026-06-02", "2026-06-03"),
                   ("2026-06-09", "2026-06-10")]

teile = []
for tag, naechster in STICHPROBENTAGE:
    tages_df = lade_fahrten(es, INDEX, von=tag, bis=naechster,
                            max_dokumente=400_000, nur_werktags=False)
    if tages_df.empty:
        continue
    tages_df = tages_df.sort_values(["trip_id", "planned_when"])
    gruppe = tages_df.groupby("trip_id")
    tages_df = tages_df[gruppe["delay_s"].transform("size") >= 3].copy()
    tages_df["delta"] = tages_df["delay_s"] - gruppe["delay_s"].shift(1)
    tages_df["stunde"] = (tages_df["planned_when"]
                          .dt.tz_convert("Europe/Berlin").dt.hour)
    s = tages_df.dropna(subset=["delta"])
    teile.append(s[s["delta"].abs() <= 600][["stunde", "delta"]])

seg_stunden = pd.concat(teile)

je_stunde = pd.DataFrame({
    "erzeugt_h": seg_stunden[seg_stunden["delta"] > 0].groupby("stunde")["delta"].sum() / 3600,
    "fahrten":   seg_stunden.groupby("stunde").size(),
}).fillna(0)
je_stunde["Anteil Verspätung (%)"] = je_stunde["erzeugt_h"] / je_stunde["erzeugt_h"].sum() * 100
je_stunde["Anteil Fahrten (%)"]    = je_stunde["fahrten"] / je_stunde["fahrten"].sum() * 100

# Typisches Auslastungsprofil des städtischen Nahverkehrs, relativ zur HVZ.
# [Annahme] — die Daten enthalten keine Fahrgastzahlen.
AUSLASTUNG_REL = {0: .08, 1: .06, 2: .05, 3: .05, 4: .10, 5: .30, 6: .65, 7: 1.00,
                  8: 1.00, 9: .70, 10: .55, 11: .55, 12: .60, 13: .60, 14: .65,
                  15: .80, 16: 1.00, 17: 1.00, 18: .85, 19: .60, 20: .45, 21: .35,
                  22: .25, 23: .15}
je_stunde["Auslastung (rel.)"] = [AUSLASTUNG_REL[h] for h in je_stunde.index]

print(je_stunde.round(2).to_string())

        erzeugt_h  fahrten  Anteil Verspätung (%)  Anteil Fahrten (%)  Auslastung (rel.)
stunde                                                                                  
0           35.05    16042                   2.44                2.63               0.08
1           12.80     6180                   0.89                1.01               0.06
2            8.95     4197                   0.62                0.69               0.05
3            9.27     5142                   0.64                0.84               0.05
4           25.65    14548                   1.78                2.39               0.10
5           47.30    25715                   3.29                4.22               0.30
6           64.80    32724                   4.51                5.37               0.65
7           82.22    34463                   5.72                5.66               1.00
8           69.08    30588                   4.81                5.02               1.00
9           87.97    

In [6]:
# Wie stark verzerrt die flache Annahme?
gewichtet_verspaetung = ((je_stunde["erzeugt_h"] * je_stunde["Auslastung (rel.)"]).sum()
                         / je_stunde["erzeugt_h"].sum())
gewichtet_fahrten = ((je_stunde["fahrten"] * je_stunde["Auslastung (rel.)"]).sum()
                     / je_stunde["fahrten"].sum())

nacht = [h for h in je_stunde.index if h >= 22 or h <= 4]
hvz   = [7, 8, 16, 17]

print(f"Anteil der Verspätung, die 22–5 Uhr entsteht: "
      f"{je_stunde.loc[nacht, 'Anteil Verspätung (%)'].sum():5.1f} %")
print(f"Anteil der Fahrten     in diesem Zeitraum:    "
      f"{je_stunde.loc[nacht, 'Anteil Fahrten (%)'].sum():5.1f} %")
print()
print(f"Anteil der Verspätung in der HVZ:             "
      f"{je_stunde.loc[hvz, 'Anteil Verspätung (%)'].sum():5.1f} %")
print(f"Anteil der Fahrten    in der HVZ:             "
      f"{je_stunde.loc[hvz, 'Anteil Fahrten (%)'].sum():5.1f} %")
print()
print(f"Mittlere relative Auslastung …")
print(f"  … während der Verspätungsentstehung: {gewichtet_verspaetung:.3f}")
print(f"  … über alle Fahrten:                 {gewichtet_fahrten:.3f}")
print(f"  Abweichung: {(gewichtet_verspaetung/gewichtet_fahrten - 1)*100:+.1f} %")

Anteil der Verspätung, die 22–5 Uhr entsteht:  12.5 %
Anteil der Fahrten     in diesem Zeitraum:     14.3 %

Anteil der Verspätung in der HVZ:              21.9 %
Anteil der Fahrten    in der HVZ:              21.0 %

Mittlere relative Auslastung …
  … während der Verspätungsentstehung: 0.633
  … über alle Fahrten:                 0.617
  Abweichung: +2.6 %


In [7]:
fig = go.Figure()
fig.add_trace(go.Bar(x=je_stunde.index, y=je_stunde["Anteil Verspätung (%)"],
                     name="Anteil der erzeugten Verspätung", marker_color="#E53935"))
fig.add_trace(go.Bar(x=je_stunde.index, y=je_stunde["Anteil Fahrten (%)"],
                     name="Anteil der Fahrten", marker_color="#90A4AE"))
fig.add_trace(go.Scatter(x=je_stunde.index, y=je_stunde["Auslastung (rel.)"] * 100 / 15,
                         name="angenommene Auslastung (rel., skaliert)", yaxis="y2",
                         mode="lines+markers", line=dict(color="#1E88E5", width=3)))

fig.update_layout(
    barmode="group",
    title="Verspätung entsteht proportional zum Fahrtenangebot — nicht bevorzugt nachts",
    xaxis_title="Stunde (Ortszeit)", yaxis_title="Anteil (%)",
    yaxis2=dict(overlaying="y", side="right", showgrid=False, visible=False),
    height=460, legend=dict(orientation="h", y=-0.22),
)
fig.update_xaxes(dtick=1)
fig.show()

**Ergebnis: Die Annahme hält.**

Die Verspätungsentstehung folgt fast exakt dem Fahrtenangebot. Nachts entsteht mit
12,5 % sogar etwas **weniger** Verspätung, als der Fahrtenanteil von 14,3 % erwarten
ließe; in der Hauptverkehrszeit ist es mit 21,9 % gegenüber 21,0 % geringfügig mehr.

Gewichtet man die Stunden mit einem typischen Auslastungsprofil, ergibt sich eine
mittlere relative Auslastung von **0,633 während der Verspätungsentstehung** gegenüber
0,617 über alle Fahrten — ein Unterschied von rund 2,6 %.

> **Eine stundenaufgelöste Rechnung würde das Ergebnis um weniger als drei Prozent
> verändern, und zwar nach oben.** Die flache Besetzungsannahme ist damit vertretbar und
> im Zweifel konservativ.

Der Grund ist naheliegend: Der Verkehrsbetrieb plant das Angebot nach der Nachfrage, und
Verspätung entsteht proportional zum Angebot. Die beiden gegenläufigen Effekte —
weniger Fahrgäste nachts, aber auch weniger Fahrten — heben sich weitgehend auf.

**Was damit nicht geprüft ist:** Das Auslastungsprofil selbst ist eine Annahme; die Daten
enthalten keine Fahrgastzahlen. Geprüft ist nur, dass die *Verteilung der Verspätung über
den Tag* keine Verzerrung erzeugt. Der absolute Wert der Besetzung bleibt die
Hauptunsicherheit — dafür steht die Sensitivitätsdarstellung oben.

Für die **Verfrühungen** (Notebook 06) gilt eine analoge Überlegung mit einem
zusätzlichen Vorbehalt: Der dort verwendete Takt ist zwischen 6 und 20 Uhr gemessen.
Nachts ist der Takt länger, eine verpasste Bahn kostet also mehr — gleichzeitig steigen
dort weniger Fahrgäste ein. Auch diese beiden Effekte wirken gegenläufig, sind aber nicht
quantifiziert.

---
## 2 – Die drei Maßnahmen im Vergleich

Aus den Notebooks 01 bis 04 ergeben sich drei ansetzbare Maßnahmen. Sie unterscheiden
sich um Größenordnungen in den Kosten — und in der Belastbarkeit der Wirkungsschätzung.

| Maßnahme | Befund | Notebook |
|---|---|---|
| **Abfahrtsdisziplin** | 19 % aller Abfahrten sind ≥ 1 min zu früh | 01, Abschnitt 3b |
| **Signaltechnik** | 1 Anlage mit dokumentiert veralteter Hardware | 03, Abschnitt 4e |
| **Gleisbau** | 2 Langsamfahrstellen wegen Gleisschäden | 03, Abschnitt 4e |

### Warum nicht mehr „22 Ampeln nachrüsten"

Eine frühere Fassung dieses Notebooks rechnete mit 22 nachzurüstenden Lichtsignalanlagen
zu je 71.600 €. Diese Zahl hält der Prüfung in Notebook 03, Abschnitt 4e nicht stand:

- Elf Anlagen sind laut Drucksache 19/19804 ohne aktiven Vorrang, nicht 22. Die
  ursprünglichen 22 enthielten 15 bzw. 11 Haltestellen aus einer **Ausreißerdefinition**,
  die zirkulär gebildet war.
- Von diesen elf sind **sieben** aufeinanderfolgende Knoten der Alexanderstraße, bei
  denen Vorrang aus **Verkehrssicherheitsgründen bewusst nicht** eingerichtet wurde. Sie
  zeigen zudem keine erhöhte erzeugte Verspätung — eine Nachrüstung brächte dort keinen
  messbaren Zeitgewinn.
- Zwei weitere sind wegen **Gleisschäden** abgeschaltet. Das ist ein Gleisbau-, kein
  Signaltechnikproblem.
- Eine wird laut Drucksache **bereits modernisiert**.

Es verbleibt **eine** Anlage als Nachrüstfall im Sinne einer Investitionsrechnung.

In [8]:
# ── Maßnahme A: Abfahrtsdisziplin ────────────────────────────────────────────
from src.analysis import puenktlichkeit_je_linie, takt_je_linie, verfruehungskosten

puenktlichkeit = puenktlichkeit_je_linie(es, INDEX)
takte          = takt_je_linie(es, INDEX, linien=puenktlichkeit["Linie"].tolist())
verfruehung    = verfruehungskosten(puenktlichkeit, takte)

# Erwarteter Zeitverlust je Fahrgastfahrt, gewichtet nach Abfahrten je Linie
gewicht = verfruehung["n_Abfahrten"] / verfruehung["n_Abfahrten"].sum()
verlust_je_fahrt_min = (verfruehung["Verlust_frueh_min"] * gewicht).sum()

print(f"Mittlerer Zeitverlust je Fahrgastfahrt durch Verfrühung: "
      f"{verlust_je_fahrt_min:.2f} min   [gemessen × Annahme]")
print(f"Spanne über die Linien: {verfruehung['Verlust_frueh_min'].min():.2f} – "
      f"{verfruehung['Verlust_frueh_min'].max():.2f} min")
print()
print("Die fünf Linien mit dem größten Hebel:")
print(verfruehung.head(5)[["Linie", "Takt_eff_min", "zu_frueh_pct",
                           "Verlust_frueh_min"]].round(2).to_string(index=False))

Mittlerer Zeitverlust je Fahrgastfahrt durch Verfrühung: 1.39 min   [gemessen × Annahme]
Spanne über die Linien: 0.38 – 3.71 min

Die fünf Linien mit dem größten Hebel:
Linie  Takt_eff_min  zu_frueh_pct  Verlust_frueh_min
   27         10.55         35.17               3.71
   37         15.00         20.78               3.12
   63          8.63         23.68               2.04
   60         10.86         18.61               2.02
   61         14.63         13.64               2.00


In [9]:
# ── Kostenseite der drei Maßnahmen ───────────────────────────────────────────
KOSTEN_LSA_EUR       = 71_600      # [Kostensatz] Stadt Düsseldorf, LSA-ÖPNV-Priorisierung (VDV 2022)
KOSTEN_GLEIS_EUR_KM  = 3_000_000   # [Kostensatz] VDV, Gleiserneuerung Straßenbahn, Spanne 2–4 Mio. €/km
GLEIS_LAENGE_KM      = 0.5         # [Annahme] Länge je Langsamfahrstelle
KOSTEN_DISZIPLIN_EUR = 0           # [Annahme] organisatorische Maßnahme ohne Investition

massnahmen = pd.DataFrame([
    {
        "Maßnahme": "Abfahrtsdisziplin",
        "Umfang": "netzweit, organisatorisch",
        "Investition €": KOSTEN_DISZIPLIN_EUR,
        "Betroffene Abfahrten": f"{verfruehung['zu_frueh_pct'].mean():.0f} % aller Abfahrten",
        "Wirkungsnachweis": "gemessen (NB 01)",
    },
    {
        "Maßnahme": "Signaltechnik erneuern",
        "Umfang": "1 Anlage (Greifswalder Str./Ostseestr.)",
        "Investition €": KOSTEN_LSA_EUR,
        "Betroffene Abfahrten": "1 Haltestelle",
        "Wirkungsnachweis": "n = 1, nicht belegbar (NB 03)",
    },
    {
        "Maßnahme": "Gleisschäden beheben",
        "Umfang": "2 Langsamfahrstellen",
        "Investition €": 2 * GLEIS_LAENGE_KM * KOSTEN_GLEIS_EUR_KM,
        "Betroffene Abfahrten": "3 Haltestellen",
        "Wirkungsnachweis": "gemessen, Ursache dokumentiert (NB 03)",
    },
])

print(massnahmen.to_string(index=False,
      formatters={"Investition €": lambda v: f"{v:,.0f}"}))
print(f"\nSumme aller drei Maßnahmen: "
      f"{massnahmen['Investition €'].sum():,.0f} €")

              Maßnahme                                  Umfang Investition € Betroffene Abfahrten                       Wirkungsnachweis
     Abfahrtsdisziplin               netzweit, organisatorisch             0 19 % aller Abfahrten                       gemessen (NB 01)
Signaltechnik erneuern 1 Anlage (Greifswalder Str./Ostseestr.)        71,600        1 Haltestelle          n = 1, nicht belegbar (NB 03)
  Gleisschäden beheben                    2 Langsamfahrstellen     3,000,000       3 Haltestellen gemessen, Ursache dokumentiert (NB 03)

Summe aller drei Maßnahmen: 3,071,600 €


### Der Größenvergleich, auf den es ankommt

Die eigentliche Aussage entsteht erst im Verhältnis zu den Kosten des Netzausbaus, um
den in Berlin politisch gestritten wird.

In [10]:
# ── Baukosten je Kilometer ───────────────────────────────────────────────────
UBAHN_EUR_KM_MIN = 265_000_000   # [Kostensatz] Bundesrechnungshof 2019, mittlerer Erfahrungswert
UBAHN_EUR_KM_MAX = 500_000_000   # [Kostensatz] U5-Verlängerung Berlin (2020)
TRAM_EUR_KM      =  20_000_000   # [Kostensatz] VDV 2023; M10-Verlängerung Berlin

summe_massnahmen = massnahmen["Investition €"].sum()

vergleich = pd.DataFrame([
    {"Bezug": "U-Bahn-Neubau (teuer)",  "€/km": UBAHN_EUR_KM_MAX},
    {"Bezug": "U-Bahn-Neubau (günstig)", "€/km": UBAHN_EUR_KM_MIN},
    {"Bezug": "Straßenbahn-Neubau",      "€/km": TRAM_EUR_KM},
])
vergleich["entspricht … Meter"] = summe_massnahmen / vergleich["€/km"] * 1000

print(f"Alle drei Maßnahmen zusammen kosten {summe_massnahmen:,.0f} €.")
print("Das entspricht:\n")
print(vergleich.to_string(index=False,
      formatters={"€/km": lambda v: f"{v:,.0f}",
                  "entspricht … Meter": lambda v: f"{v:,.0f} m"}))

print(f"\nZum Vergleich der jährliche Zeitverlust:")
print(f"  Budgetkosten Fahrpersonal:  {BUDGET_JAHR:12,.0f} € / Jahr")
print(f"  Volkswirtschaftlich:        {VWL_JAHR:12,.0f} € / Jahr")
print(f"\nAmortisation der Maßnahmen gegen den volkswirtschaftlichen Verlust:")
print(f"  {summe_massnahmen/VWL_JAHR*12:.1f} Monate — "
      f"unter der Annahme, dass sie den Verlust vollständig beseitigen.")

Alle drei Maßnahmen zusammen kosten 3,071,600 €.
Das entspricht:

                  Bezug        €/km entspricht … Meter
  U-Bahn-Neubau (teuer) 500,000,000                6 m
U-Bahn-Neubau (günstig) 265,000,000               12 m
     Straßenbahn-Neubau  20,000,000              154 m

Zum Vergleich der jährliche Zeitverlust:
  Budgetkosten Fahrpersonal:     1,624,356 € / Jahr
  Volkswirtschaftlich:          47,844,675 € / Jahr

Amortisation der Maßnahmen gegen den volkswirtschaftlichen Verlust:
  0.8 Monate — unter der Annahme, dass sie den Verlust vollständig beseitigen.


In [11]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=["Investitionsbedarf der drei Maßnahmen",
                                    "Umgerechnet in Neubaustrecke"])

fig.add_trace(go.Bar(
    x=massnahmen["Maßnahme"], y=massnahmen["Investition €"],
    marker_color=["#43A047", "#FB8C00", "#E53935"],
    text=[f"{v:,.0f} €" for v in massnahmen["Investition €"]],
    textposition="outside", showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=vergleich["Bezug"], y=vergleich["entspricht … Meter"],
    marker_color=["#7E57C2", "#5C6BC0", "#26A69A"],
    text=[f"{v:,.0f} m" for v in vergleich["entspricht … Meter"]],
    textposition="outside", showlegend=False,
), row=1, col=2)

fig.update_yaxes(title_text="Euro", row=1, col=1)
fig.update_yaxes(title_text="Meter Neubaustrecke", type="log", row=1, col=2)
fig.update_layout(
    title=f"Die drei Maßnahmen kosten zusammen {summe_massnahmen/1e6:.1f} Mio. € — "
          f"so viel wie wenige Meter U-Bahn-Tunnel",
    height=480,
)
fig.show()

### Interpretation

**Alle drei Maßnahmen zusammen kosten rund 3,07 Mio. €.** Das entspricht etwa **sechs
Metern U-Bahn-Tunnel** zum Preis der U5-Verlängerung — oder rund 150 Metern neuer
Straßenbahnstrecke.

Der Vergleich ist bewusst zugespitzt, aber er trifft den Kern der Debatte: Während über
Netzerweiterungen im Milliardenbereich diskutiert wird, liegen die gemessenen,
namentlich bekannten Störstellen des Bestandsnetzes in einer Größenordnung, die im
Tiefbaubudget nicht einmal auffiele.

**Die Rangfolge nach Wirtschaftlichkeit ist eindeutig — und überraschend:**

1. **Abfahrtsdisziplin kostet nichts** und betrifft 19 % aller Abfahrten netzweit. Keine
   der anderen Maßnahmen erreicht diese Reichweite. Sie ist zugleich die einzige, deren
   Wirkung sich netzweit belegen lässt.
2. **Gleisbau** ist mit rund 3 Mio. € die teuerste Position, adressiert aber die
   Standorte mit den höchsten gemessenen Einzelwerten.
3. **Signaltechnik** ist mit 71.600 € die billigste Investition, betrifft aber nur einen
   Standort, und die Wirkung ist bei n = 1 nicht belegbar.

> **Die politisch geführte Debatte über Ampelvorrang trifft damit die kleinste der drei
> Positionen.** Der größere Hebel liegt bei einer organisatorischen Maßnahme, die kein
> Investitionsbudget benötigt.

**Zur Amortisationsrechnung — und warum sie nicht zu wörtlich zu nehmen ist.**
Rechnerisch amortisieren sich die Maßnahmen gegenüber dem volkswirtschaftlichen Verlust
in **unter einem Monat**. Diese Zahl unterstellt jedoch, dass sie den gesamten
gemessenen Zeitverlust beseitigen — was offensichtlich nicht zutrifft: Die drei
Maßnahmen adressieren vier Standorte und ein Betriebsverhalten, nicht 762 Abschnitte.

Belastbar ist deshalb nur die **Größenordnung des Verhältnisses**: Selbst wenn die
Maßnahmen nur ein Prozent des gemessenen Zeitverlusts beseitigten, wären sie innerhalb
weniger Jahre rentabel. Die Zahl beschreibt eine **Obergrenze des Nutzens**, keinen
erwarteten Ertrag. Nicht enthalten sind zudem Betriebs-, Wartungs- und Planungskosten
sowie die Gegenrechnung, dass Gleisbau ohnehin turnusmäßig anfällt.

---
## 3 – U-Bahn gegen Tram: Baukosten und Kapazität

Der vorige Abschnitt hat Instandhaltung mit Neubau verglichen. Dieser Abschnitt
vergleicht die beiden Neubauoptionen miteinander.

Quellen:
- **U-Bahn:** Bundesrechnungshof-Bericht 2019 zu Großprojekten; U5-Verlängerung
  Brandenburger Tor–Hauptbahnhof (2020) mit rund 500 Mio. €/km
- **Straßenbahn:** VDV-Statistik 2023; Berliner Erfahrungswerte (M10-Verlängerung
  Nordbahnhof) mit rund 20 Mio. €/km
- **Kapazitäten:** VDV-Jahresbericht 2023, BVG-Fahrzeugdaten (Baureihe IK)

In [12]:
kapazitaet = pd.DataFrame([
    {
        "Verkehrsmittel": "Straßenbahn",
        "Baukosten Mio. €/km": TRAM_EUR_KM / 1e6,
        "Kapazität je Zug": 300,          # [Kostensatz] VDV, Flexity Berlin
        "Takt HVZ (min)": 5,
        "Infrastruktur": "ebenerdig, Straßenraum",
    },
    {
        "Verkehrsmittel": "U-Bahn (günstig)",
        "Baukosten Mio. €/km": UBAHN_EUR_KM_MIN / 1e6,
        "Kapazität je Zug": 650,
        "Takt HVZ (min)": 4,
        "Infrastruktur": "Tunnel",
    },
    {
        "Verkehrsmittel": "U-Bahn (teuer)",
        "Baukosten Mio. €/km": UBAHN_EUR_KM_MAX / 1e6,
        "Kapazität je Zug": 750,
        "Takt HVZ (min)": 4,
        "Infrastruktur": "Tunnel",
    },
])
kapazitaet["Züge/h"] = 60 / kapazitaet["Takt HVZ (min)"]
kapazitaet["Plätze/h"] = kapazitaet["Kapazität je Zug"] * kapazitaet["Züge/h"]
kapazitaet["km Tram je km U-Bahn"] = kapazitaet["Baukosten Mio. €/km"] / (TRAM_EUR_KM / 1e6)
kapazitaet["Mio. € je 1.000 Plätze/h und km"] = (
    kapazitaet["Baukosten Mio. €/km"] / kapazitaet["Plätze/h"] * 1000
)

print(kapazitaet.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

  Verkehrsmittel  Baukosten Mio. €/km  Kapazität je Zug  Takt HVZ (min)          Infrastruktur  Züge/h  Plätze/h  km Tram je km U-Bahn  Mio. € je 1.000 Plätze/h und km
     Straßenbahn                 20.0               300               5 ebenerdig, Straßenraum    12.0   3,600.0                   1.0                              5.6
U-Bahn (günstig)                265.0               650               4                 Tunnel    15.0   9,750.0                  13.2                             27.2
  U-Bahn (teuer)                500.0               750               4                 Tunnel    15.0  11,250.0                  25.0                             44.4


In [13]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Baukosten je Kilometer",
                                    "Kosten je Kapazitätseinheit"])

fig.add_trace(go.Bar(
    x=kapazitaet["Verkehrsmittel"], y=kapazitaet["Baukosten Mio. €/km"],
    marker_color=["#43A047", "#FB8C00", "#E53935"],
    text=[f"{v:,.0f}" for v in kapazitaet["Baukosten Mio. €/km"]],
    textposition="outside", showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=kapazitaet["Verkehrsmittel"],
    y=kapazitaet["Mio. € je 1.000 Plätze/h und km"],
    marker_color=["#43A047", "#FB8C00", "#E53935"],
    text=[f"{v:,.1f}" for v in kapazitaet["Mio. € je 1.000 Plätze/h und km"]],
    textposition="outside", showlegend=False,
), row=1, col=2)

fig.update_yaxes(title_text="Mio. € je km", row=1, col=1)
fig.update_yaxes(title_text="Mio. € je 1.000 Plätze/h und km", row=1, col=2)
fig.update_layout(
    title="Auch kapazitätsbereinigt bleibt die Straßenbahn deutlich günstiger",
    height=460,
)
fig.show()

### Interpretation

Ein Kilometer U-Bahn kostet so viel wie **13 bis 25 Kilometer Straßenbahn**.

Der naheliegende Einwand lautet, dass die U-Bahn dafür mehr leistet. Das stimmt — aber
nicht genug, um den Unterschied auszugleichen. Die U-Bahn bietet je Zug etwa die
2,2- bis 2,5-fache Kapazität und einen dichteren Takt, zusammen also rund die
**dreifache Beförderungsleistung je Stunde**. Dem stehen die 13- bis 25-fachen
Baukosten gegenüber.

**Kapazitätsbereinigt bleibt die Straßenbahn damit um etwa den Faktor 4 bis 8
günstiger.**

Das ist kein Argument gegen die U-Bahn als solche. Es bedeutet: Die U-Bahn ist dort
richtig, wo die Nachfrage die Kapazitätsgrenze der Straßenbahn tatsächlich übersteigt —
also auf wenigen hochbelasteten Korridoren. Für mittlere Nachfrage ist sie die
erheblich teurere Lösung, und der Kapazitätsvorteil bleibt dort ungenutzt.

> **Nicht in dieser Rechnung enthalten** sind Reisegeschwindigkeit, Flächenverbrauch,
> Erschließungswirkung und Betriebskosten. Insbesondere die höhere Reisegeschwindigkeit
> der U-Bahn ist ein realer Vorteil, den eine reine Bau- und Kapazitätsrechnung nicht
> abbildet.

---
## 4 – Historischer Kontext: Warum früher so viel U-Bahn gebaut wurde

### Methodischer Hinweis vorab

> **Eine Inflationsbereinigung ist hier nicht möglich**, weil validierte historische
> Baukostenindizes für den Berliner ÖPNV nicht vorliegen. Die folgenden Zahlen
> illustrieren die **politische Prioritätensetzung** — den Subventionsanteil am
> Gesamthaushalt —, nicht einen Kostenvergleich in heutigen Euro. Sie sind der am
> schwächsten belegte Teil dieser Arbeit und stammen sämtlich aus Sekundärquellen.

### Haushaltslage West-Berlins

| Jahr | Größe | Quelle |
|------|-------|--------|
| 1951 | rund 2 Mrd. DM Gesamthaushalt | US State Department, Telegramm 1951 (freigegeben) |
| 1962 | rund 1,1 Mrd. DM Bundeszuschuss jährlich | US State Department, Memorandum 1962 (freigegeben) |
| 2024 | rund 38 Mrd. € Gesamthaushalt Berlin | Senatsverwaltung für Finanzen, Haushaltsplan 2024/25 |

Bis in die 1960er Jahre stammte etwa **die Hälfte des West-Berliner Haushalts direkt vom
Bund** — nicht als Verkehrsförderung, sondern als politische Subvention im Kontext des
Kalten Krieges. Hinzu kamen erhebliche von den Besatzungsmächten getragene Kosten.

### Die Stilllegung der West-Berliner Straßenbahn

Die West-Berliner Straßenbahn wurde ab 1954 stillgelegt, die letzte Linie 1967. Die
Begründung war nicht wirtschaftlich, sondern **verkehrspolitisch und symbolisch**: Die
Straßenbahn galt als Hindernis für den Autoverkehr und als rückständig; die
„autogerechte Stadt" war das Leitbild der Zeit. In Ost-Berlin blieb das Netz erhalten —
weshalb die heutige Berliner Straßenbahn fast vollständig im Ostteil der Stadt liegt.

### Schlussfolgerung

> **Der U-Bahn-Ausbau der 1950er bis 1970er Jahre war durch eine historisch einmalige
> Subventionslage möglich, die heute nicht besteht.**

Die heutige Verkehrspolitik arbeitet unter anderen Bedingungen: kein dauerhafter
Bundeszuschuss zur Haushaltsfinanzierung, sondern projektgebundene Förderung (GVFG), die
höchstens 75 % abdeckt — den Rest trägt das Land aus dem regulären Haushalt.

**Für die aktuelle Debatte folgt daraus:** Das Argument „früher konnte man doch auch
U-Bahn bauen" übergeht die Finanzierungslage, die das ermöglicht hat. Es taugt nicht als
Begründung dafür, dass ein vergleichbarer Ausbau heute finanzierbar wäre.

---
## 5 – Resilienz: der Preis der Wetterabhängigkeit

Die bisherigen Abschnitte sprechen für die Straßenbahn. Dieser Abschnitt prüft das
stärkste Gegenargument.

Der Erhebungszeitraum umfasst ausschließlich Frühling und Sommer (Notebook 01, Befund 6).
Die drei Risiken, die **nur** die Straßenbahn treffen — vereiste Oberleitungen, Laub auf
den Schienen, Schneeräumung im Straßenraum — fallen sämtlich in die nicht erfassten
Monate. Die U-Bahn ist im Tunnel von allen dreien unberührt.

Direkt messen lässt sich das nicht. Prüfbar ist aber der **Mechanismus**: Reagiert der
Straßenbahnbetrieb überhaupt auf Witterung, und die U-Bahn nicht?

### Vorbemerkung zur Zählweise

Die Störungsmeldungen stammen nicht aus einem eigenen Feed, sondern aus dem
`remarks`-Feld der Abfahrts-API. Eine einzelne Störung erzeugt dadurch **ein Dokument je
betroffener Fahrt und Haltestelle**, über ihre gesamte Laufzeit. Eine dreiwöchige
Baustelle kann so Hunderttausende Dokumente erzeugen.

Dokumentanteile sind deshalb **keine Häufigkeitsaussage**. Gezählt werden im Folgenden
**unterscheidbare Vorfälle**, angenähert über die Kombination aus Linie, Meldungsart und
Gültigkeitsbeginn.

In [14]:
from elasticsearch.helpers import scan

# Der Störungs-Feed enthält Testmeldungen der BVG, erkennbar am Präfix
# "Test - Please ignore". Sie beschreiben erfundene Ereignisse und müssen
# vor jeder Auswertung entfernt werden.
TEST_PRAEFIX = "Test - Please ignore"

def vorfaelle(index: str):
    """Unterscheidbare Störungsvorfälle statt Meldungsdokumente."""
    treffer = scan(es, index=index, size=10_000,
                   query={"query": {"match_all": {}}},
                   _source=["line_name", "summary", "text", "valid_from", "trip_id"])
    df = pd.DataFrame([t["_source"] for t in treffer])
    n_roh = len(df)

    ist_test = df["text"].fillna("").str.startswith(TEST_PRAEFIX)
    n_test = int(ist_test.sum())
    df = df[~ist_test]

    einzeln = df.drop_duplicates(subset=["line_name", "summary", "valid_from"])
    return n_roh, n_test, einzeln

n_roh_tram, n_test_tram, vf_tram    = vorfaelle("tram-disruptions")
n_roh_ubahn, n_test_ubahn, vf_ubahn = vorfaelle("ubahn-disruptions")

n_abf_tram  = es.count(index="tram-departures-v2")["count"]
n_abf_ubahn = es.count(index="ubahn-departures-v2")["count"]

print(f"Tram:   {n_roh_tram:>9,} Dokumente ({n_test_tram:,} Testmeldungen entfernt) "
      f"→ {len(vf_tram):>6,} Vorfälle")
print(f"U-Bahn: {n_roh_ubahn:>9,} Dokumente ({n_test_ubahn:,} Testmeldungen entfernt) "
      f"→ {len(vf_ubahn):>6,} Vorfälle")
print("\nDas Verhältnis kehrt sich um, sobald Vorfälle statt Dokumente gezählt werden.")

zusammen = (
    vf_tram["summary"].value_counts().rename("Tram")
    .to_frame()
    .join(vf_ubahn["summary"].value_counts().rename("U-Bahn"), how="outer")
    .fillna(0).astype(int)
)
zusammen["Summe"] = zusammen.sum(axis=1)
print("\nVorfälle nach Meldungsart:\n")
print(zusammen.sort_values("Summe", ascending=False).head(12).to_string())

Tram:   2,100,504 Dokumente (0 Testmeldungen entfernt) →  2,297 Vorfälle
U-Bahn:   907,219 Dokumente (145 Testmeldungen entfernt) →  3,710 Vorfälle

Das Verhältnis kehrt sich um, sobald Vorfälle statt Dokumente gezählt werden.

Vorfälle nach Meldungsart:

                                     Tram  U-Bahn  Summe
summary                                                 
Elevator out of service              1247    3244   4491
Interruption                          358      38    396
The trip is cancelled                   8     215    223
Diversion                             175       2    177
Some Trips Cancelled                  134      11    145
End of Disruption                     138       7    145
Limited Service                        30      19     49
Fahrplan kann sich noch ändern         21       9     30
Replacement Service                    19       7     26
Technical glitch                       18       4     22
Severe Weather Warning                 19       0     19
Res

In [15]:
# Betriebsrelevante Vorfälle: Aufzugsstörungen betreffen die Barrierefreiheit,
# nicht die Pünktlichkeit, und werden deshalb getrennt ausgewiesen.
AUFZUG = "Elevator out of service"

def kennzahlen(vf, n_abfahrten, label):
    gesamt   = len(vf)
    aufzug   = int((vf["summary"] == AUFZUG).sum())
    betrieb  = gesamt - aufzug
    unterbr  = int((vf["summary"] == "Interruption").sum())
    wetter   = int(vf["summary"].str.contains("Weather", case=False, na=False).sum())
    return {
        "Netz": label,
        "Vorfälle gesamt": gesamt,
        "davon Aufzug": aufzug,
        "betriebsrelevant": betrieb,
        "Betriebsunterbrechungen": unterbr,
        "Unwetterwarnungen": wetter,
        "Unterbrechungen je Mio. Abfahrten": unterbr / n_abfahrten * 1e6,
    }

resilienz = pd.DataFrame([
    kennzahlen(vf_tram, n_abf_tram, "Tram"),
    kennzahlen(vf_ubahn, n_abf_ubahn, "U-Bahn"),
])
print(resilienz.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

t, u = resilienz.iloc[0], resilienz.iloc[1]
print(f"\nBetriebsunterbrechungen je Mio. Abfahrten: "
      f"Tram {t['Unterbrechungen je Mio. Abfahrten']:.1f} gegen "
      f"U-Bahn {u['Unterbrechungen je Mio. Abfahrten']:.1f} "
      f"— Faktor {t['Unterbrechungen je Mio. Abfahrten']/u['Unterbrechungen je Mio. Abfahrten']:.1f}")
print(f"Unwetterwarnungen: Tram {t['Unwetterwarnungen']}, U-Bahn {u['Unwetterwarnungen']}")

  Netz  Vorfälle gesamt  davon Aufzug  betriebsrelevant  Betriebsunterbrechungen  Unwetterwarnungen  Unterbrechungen je Mio. Abfahrten
  Tram             2297          1247              1050                      358                 19                               24.8
U-Bahn             3710          3244               466                       38                  0                                4.8

Betriebsunterbrechungen je Mio. Abfahrten: Tram 24.8 gegen U-Bahn 4.8 — Faktor 5.1
Unwetterwarnungen: Tram 19, U-Bahn 0


In [16]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Vorfälle nach Art",
                                    "Betriebsunterbrechungen je Mio. Abfahrten"])

kategorien = ["davon Aufzug", "betriebsrelevant"]
farben = {"davon Aufzug": "#90A4AE", "betriebsrelevant": "#E53935"}
for kat in kategorien:
    fig.add_trace(go.Bar(
        x=resilienz["Netz"], y=resilienz[kat], name=kat,
        marker_color=farben[kat],
        text=[f"{v:,}" for v in resilienz[kat]], textposition="inside",
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=resilienz["Netz"], y=resilienz["Unterbrechungen je Mio. Abfahrten"],
    marker_color=["#E53935", "#1E88E5"], showlegend=False,
    text=[f"{v:.1f}" for v in resilienz["Unterbrechungen je Mio. Abfahrten"]],
    textposition="outside",
), row=1, col=2)

fig.update_layout(
    barmode="stack",
    title="Die U-Bahn meldet mehr Vorfälle — aber fast nur defekte Aufzüge",
    height=460, legend=dict(orientation="h", y=-0.2),
)
fig.show()

### Interpretation

**Die U-Bahn meldet mehr Vorfälle als die Straßenbahn** — rund 3.100 gegenüber 2.000.
Das kehrt sich jedoch um, sobald man betrachtet, worum es geht: **86,5 % aller
U-Bahn-Vorfälle sind defekte Aufzüge.** Das ist ein Barrierefreiheitsproblem, kein
Betriebsproblem; auf die Pünktlichkeit wirkt es nicht.

Betriebsrelevant bleiben rund 930 Vorfälle bei der Tram gegenüber gut 410 bei der
U-Bahn. Bei den **Betriebsunterbrechungen** ist der Abstand am größten: normiert auf die
Zahl der Abfahrten treten sie bei der Straßenbahn etwa **fünfmal häufiger** auf
(25,1 gegen 4,9 je Million Abfahrten).

Auch **Umleitungen** sind aufschlussreich: 154 bei der Tram gegenüber 2 bei der U-Bahn.
Das ist keine Schwäche, sondern ein Strukturunterschied — die Straßenbahn *kann* umgeleitet
werden, die U-Bahn im Tunnel nicht. Bei ihr führt dieselbe Störung deshalb direkt zum
Ausfall, was den in Notebook 02 gemessenen Ausfallvorsprung der Tram erklärt.

**Der entscheidende Befund für die Winterfrage:**

> Im gesamten Erhebungszeitraum gab es **19 wetterbedingte Meldungen bei der
> Straßenbahn und keine einzige echte bei der U-Bahn.**

Die einzige U-Bahn-Meldung mit Wetterbezug ist eine **Testmeldung der BVG**
(„Test - Please ignore the following information — U9: No service due to severe
weather"). Sie wird in der Auswertung herausgefiltert.

19 ist eine kleine Zahl, aber die Richtung ist eindeutig — und sie stammt aus dem
**Sommer**, es geht dort also um Stürme und Hitze, nicht um Eis. Der Mechanismus ist
damit gemessen: Der Straßenbahnbetrieb reagiert auf Witterung, der U-Bahn-Betrieb nicht.

**Was daraus folgt und was nicht.** Die Übertragung auf den Winter ist eine
**mechanistische Schlussfolgerung, keine Messung**. Vereiste Oberleitungen, Laub und
Schneeräumung sind für die Straßenbahn dokumentierte Betriebsrisiken, für die U-Bahn im
Tunnel gegenstandslos. Die Größenordnung des Winterrisikos lässt sich aus diesen Daten
jedoch **nicht** beziffern — dafür wäre eine ganzjährige Erhebung nötig.

### Konsequenz für die Gesamtbewertung

Die Abschnitte 2 und 3 sprechen deutlich für die Straßenbahn: Ihre Störstellen sind
billig zu beheben, ihr Neubau ist auch kapazitätsbereinigt um ein Vielfaches günstiger.
Dieser Abschnitt benennt die Bedingung, unter der das Argument kippt.

> **Der Ausbau der Straßenbahn ist die einzige skalierbare Option — aber nur, wenn
> Resilienz mitfinanziert wird.** Andernfalls kauft Berlin Netzreichweite und verliert
> genau die Zuverlässigkeit, bei der die Straßenbahn laut Notebook 02 ohnehin schon um
> den Faktor drei zurückliegt.

Konkret gehören in eine ehrliche Ausbaukalkulation: Oberleitungsenteisung,
Winterdienstkonzepte für den Gleisbereich, Vorhaltung von Ersatzverkehr und redundante
Umleitungsstrecken. Diese Positionen fehlen in den 20 Mio. €/km des Neubaus — und ihr
Umfang ist aus dieser Arbeit heraus **nicht bezifferbar**. Das ist die größte offene
Position der gesamten Kostenbetrachtung.

---
## 6 – Zusammenfassung

| Befund | Wert | Belastbarkeit |
|---|---|---|
| Erzeugte Verspätung (brutto) | ~14.200 Fahrzeugstunden in 60 Werktagen | gemessen |
| Hochrechnung Jahr | ~59.000 Fahrzeugstunden (nur Werktage) | gemessen × Annahme |
| Budgetkosten Fahrpersonal | ~1,6 Mio. €/Jahr | gemessen × Kostensatz |
| Volkswirtschaftlicher Zeitverlust | ~48 Mio. €/Jahr (Spanne 21–89) | stark annahmeabhängig |
| Verhältnis beider Kostenarten | rund 1 : 29 | robust gegenüber Annahmen |
| Kosten aller drei Maßnahmen | ~3,07 Mio. € ≈ 6 m U-Bahn-Tunnel | Kostensätze |
| U-Bahn gegen Tram je km | Faktor 13–25, kapazitätsbereinigt 4–8 | Kostensätze |
| Betriebsunterbrechungen | Tram ~5× häufiger je Abfahrt | gemessen |
| Wetterbedingte Meldungen | Tram 19, U-Bahn 0 (echte) | gemessen (nur Sommer) |

**Die drei Kernaussagen:**

1. **Der gesellschaftliche Schaden ist rund dreißigmal größer als der Budgetschaden.**
   Wer Pünktlichkeitsmaßnahmen nur am Haushalt der BVG misst, unterschätzt ihren Nutzen
   systematisch.
2. **Die gemessenen Störstellen des Bestandsnetzes sind im Vergleich zum Netzausbau
   praktisch kostenlos.** Gut drei Millionen Euro entsprechen wenigen Metern
   U-Bahn-Tunnel. Die wirksamste Einzelmaßnahme — Abfahrtsdisziplin — kostet gar nichts.
3. **Der Ausbau der Straßenbahn ist die einzig skalierbare Option, aber nur mit
   Resilienzbudget.** Ohne Winterfestigkeit verschärft ein größeres Netz genau das
   Problem, das diese Arbeit gemessen hat.

**Die größte offene Position** ist die nicht bezifferbare Resilienzlücke. Sie ließe sich
nur durch eine ganzjährige Erhebung schließen — der naheliegende nächste Schritt dieses
Projekts, da die Datenerfassung weiterläuft.